# 🧩 Prerequisites

In [ ]:
# @title 📀 Mount Google Drive
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
%cd /content/drive/MyDrive/Colab/thesis/rrs
CWD = Path.cwd()

In [ ]:
# @title 📦 Package Installation
%%capture

# Install required packages
!pip install uv
!uv pip install unsloth

# Import necessary libraries
import os
import re
import torch

# 📊 Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "data/train2-xpo2.jsonl",
})
train_dataset = dataset["train"]

print(dataset)

# 🚀 Train Model

In [ ]:
# @title 📦 Model Loading and Configuration
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="models/qwen3-sft-lora",
    max_seq_length=2048,
    load_in_4bit=True
)

In [ ]:
system_prompt = """You are a Medical Radiology Hallucination Detector and Corrector."""
user_prompt = """\
# Instructions:
1. Compare the [RADIOLOGY FINDINGS] with its [RADIOLOGY IMPRESSION].
2. Identify the incorrect detail in the [RADIOLOGY IMPRESSION] and fix it using the [RADIOLOGY FINDINGS].
3. Output ONLY the final [CORRECTED RADIOLOGY IMPRESSION] with minimal text.

# Context:
[RADIOLOGY FINDINGS]: {}
[RADIOLOGY IMPRESSION]: {}

# Output:
[CORRECTED RADIOLOGY IMPRESSION]:"""

# clean_pattern = re.compile(r"(?:\n|/\\|\\/|\s+)")

def format_prompt(batch):
    prompts   = []
    chosens   = []
    rejecteds = []

    sources   = batch["findings"]
    targets   = batch["biobart_gen"]
    accepteds = batch["impression"]
    hallucins = batch["hallcn"]

    for source, target, accepted, rejected in zip(sources, targets, accepteds, hallucins):
        # source = clean_pattern.sub(" ", source).strip()
        formatted_user_prompt = user_prompt.format(source, target)

        prompts.append([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": formatted_user_prompt}
        ])

        chosens.append([
            {"role": "assistant", "content": accepted}
        ])

        rejecteds.append([
            {"role": "assistant", "content": rejected}
        ])

    batch["prompt"] = prompts
    batch["chosen"] = chosens
    batch["rejected"] = rejecteds

    return batch


train_dataset = train_dataset.map(format_prompt, batched=True)
train_dataset = train_dataset.remove_columns(["findings", "impression", "hallcn", "biobart_gen"])

In [ ]:
from pprint import pprint

row = train_dataset[0]
print("INSTRUCTION: " + "=" * 120)
pprint(row["prompt"], width=160)

print("ACCEPTED: " + "=" * 120)
pprint(row["chosen"], width=160)

print("REJECTED: " + "=" * 120)
pprint(row["rejected"], width=160)

## 🚀 Model Training Setup

In [ ]:
from trl.trainer.cpo_trainer import CPOConfig, CPOTrainer

simpo_trainer = CPOTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=CPOConfig(
        loss_type="simpo",
        cpo_alpha=0.05,
        simpo_gamma=5.4,
        beta=10,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=5e-6,
        logging_steps=10,
        output_dir="/content/outputs",
        eval_strategy="no",
        save_strategy="no",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        bf16=True, tf32=True,
        seed=3407,
        report_to="none"
    ),
)

In [ ]:
print(tokenizer.decode(simpo_trainer.train_dataset[0]["chosen_input_ids"]))

In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
simpo_trainer.train()

## 💾 Model Saving and Export

In [ ]:
model.save_pretrained("models/qwen3-simpo")
tokenizer.save_pretrained("models/qwen3-simpo")

In [ ]:
model.save_pretrained_merged(
    "/content/qwen3-simpo-merged",
    tokenizer, save_method = "merged_16bit"
)

In [ ]:
!ls -lh /content/qwen3-simpo-merged/

In [ ]:
!uv pip install ctranslate2

In [ ]:
!ct2-transformers-converter --model /content/qwen3-simpo-merged --output_dir models/qwen3-simpo-merged-ct2 --quantization int8_float16

In [ ]:
!cp /content/qwen3-simpo-merged/chat_template.jinja /content/qwen3-simpo-merged/tokenizer.json /content/qwen3-simpo-merged/tokenizer_config.json models/qwen3-simpo-merged-ct2/

In [ ]:
!ls -lh models/qwen3-simpo-merged-ct2